In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [ ]:
import warnings

# Ignore all warnings
warnings.filterwarnings('ignore')
# Set the path to the CSV file within the dataset
file_path = "data.csv"

# Load the latest version of the Breast Cancer Wisconsin dataset into a pandas DataFrame
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "uciml/breast-cancer-wisconsin-data",
    file_path,
    pandas_kwargs={"sep": ","}   # adjust if needed; by default pandas.read_csv parameters
)

# Display the first 5 records
print("First 5 records:\n", df.head())


In [ ]:
df.info()

In [ ]:
df['diagnosis'].value_counts()

In [ ]:
lb= LabelEncoder()
df['diagnosis']=lb.fit_transform(df['diagnosis'])
df.head()

In [ ]:
df.drop(['id','Unnamed: 32'],axis=1,inplace=True)
df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
st = StandardScaler()

X= df.drop('diagnosis',axis=1)
y= df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train_scaled = st.fit_transform(X_train)
X_test_scaled = st.transform(X_test)



In [ ]:
model = Sequential([
    Dense(128,activation='relu',input_shape=(30,)),
    Dropout(0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid'),
])

In [ ]:
model.compile(
    optimizer='adam',
    loss = 'BinaryCrossentropy',
    metrics = ['accuracy']
)

In [ ]:
early_stop = EarlyStopping(
    monitor = 'val_loss',
    patience=5,
    mode='min',
    restore_best_weights=True,
)

In [ ]:
his = model.fit(
    X_train_scaled,y_train,
    epochs = 30,
    validation_split=0.2,
    callbacks=[early_stop]
)

In [ ]:
results = model.evaluate(X_test_scaled, y_test, batch_size=32)
print(dict(zip(model.metrics_names, results)))

In [ ]:
import matplotlib.pyplot as plt

# Extract
acc = his.history['accuracy']
val_acc = his.history['val_accuracy']
loss = his.history['loss']
val_loss = his.history['val_loss']

# Plot accuracy
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.legend()
plt.title('Accuracy Over Epochs')
plt.show()

# Plot loss
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.legend()
plt.title('Loss Over Epochs')
plt.show()
